# 02 — A hybrid neural density model with a linear scale nuisance

We now build the **good fitted model**, before deliberately introducing misspecification in notebook 03. There is no event selection. The five reconstructed observables $x=(x_1,\ldots,x_5)$ enter the density networks; physical parameters enter the likelihood later.

The reference design is the same as Exercise 5: an equally weighted nominal signal/background mixture, learned by a normalized flow $p_{\mathrm{ref}}(x)$. Its 50:50 training composition is a numerical choice. The experiment still has $S=100$ and $B=10\,000$ at $\mu=1$.

We train nominal process/reference ratios and up/down process/nominal ratios, normalize the resulting six complete anchors, and use **linear interpolation of normalized densities**. This gives shape-only nuisance variations without a parameter-dependent normalization integral.

Run notebook 01 first. The defaults preserve the previous training sizes and architectures. Full training requires a GPU and substantial memory; this notebook does not claim good training until the held-out diagnostics below have been inspected.

## Setup and shared run directory

Run this cell first. Use the same `RUN_NAME` in all five notebooks. Colab can keep the run on Drive; locally, open Jupyter in this directory. Use a fresh kernel when switching from the older exercises. Training and samples remain in the run directory.


In [ ]:
# Run this cell first in each notebook. All five notebooks share RUN_NAME.
import os
import sys
import subprocess
from pathlib import Path

RUN_NAME = "default"  # Use a new name when changing the model or training setup.
USE_DRIVE = True
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO = Path("/content/nsbi_calibration_toolkit")
    BRANCH = "ml4hep_school_tutorial"
    if not (REPO / ".git").exists():
        subprocess.run([
            "git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
            "--branch", BRANCH,
            "https://github.com/rafaellopesdesa/nsbi-lhc-toolkit.git", str(REPO),
        ], check=True, env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"})
        subprocess.run([
            "git", "-C", str(REPO), "sparse-checkout", "set", "src",
            "workshops/ml4hep_tifr_colab/calibration",
        ], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH], check=True)
    SOURCE = REPO / "workshops/ml4hep_tifr_colab/calibration"
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q", "-r",
        str(SOURCE / "requirements.txt"),
    ], check=True)
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        RUN_BASE = Path("/content/drive/MyDrive/Colab Notebooks/ml4hep_tifr_colab/calibration_runs")
    else:
        RUN_BASE = Path("/content/calibration_runs")
else:
    # Open the notebook with calibration/ as its working directory.
    SOURCE = Path.cwd().resolve()
    REPO = SOURCE.parents[2]
    RUN_BASE = SOURCE / "runs"

sys.path.insert(0, str(REPO / "src"))
sys.path.insert(0, str(SOURCE))
os.chdir(SOURCE)
RUN = Path(os.environ.get("CALIBRATION_RUN", RUN_BASE / RUN_NAME)).resolve()
RUN.mkdir(parents=True, exist_ok=True)
print("Source:", SOURCE)
print("Run:", RUN)
print("Use a fresh kernel when switching between the old exercises and calibration.")


In [ ]:
import gc
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from model import FEATURES, load_config, morph, nll, fit, asimov_weights
from sampling import read_sample
from flow_reference import train_reference, MODEL_DEFAULTS, TRAIN_DEFAULTS
from utils import (HybridModel, RatioEnsemble, train_ratio, save_hybrid,
                   plot_ratio_diagnostics, plot_shape_closure, RATIO_DEFAULTS)

config = load_config(RUN)
SEED = config['seed'] + 100
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
HYBRID = RUN / 'hybrid'
PLOTS = HYBRID / 'plots'
PLOTS.mkdir(parents=True, exist_ok=True)
print('Device:', DEVICE)
print('Sample configuration:', config['samples'])

## 1. Fix the sample budgets and the model definitions

| Stage | Input sample size | Network |
|---|---:|---|
| Reference flow | 500,000 nominal events per process | 10 spline coupling layers, 16 bins, tail bound 5; 4-block networks of width 1024 |
| Nominal signal/reference and background/reference ratios | 5,000,000 events per class | Four independently initialized members, 4 hidden layers of width 1024 |
| Each up/down to nominal deformation ratio | 1,000,000 events per class | One network, 4 hidden layers of width 1024 |
| Anchor normalization | 5,000,000 fresh reference-flow events | No training |
| External diagnostics | Up to 250,000 independent events per process/anchor | No training or tuning |

The first three sizes come from notebook 01's configuration, so a reduced run remains internally consistent. The flow and nominal ratio training partitions are disjoint. Scale variations share latent events with their nominal partners; the final diagnostics use an entirely separate row block. For the deformation classifiers we use the first 1,000,000 varied rows and the last 1,000,000 nominal ratio-training rows, so paired latent events do not straddle their internal splits at the default sizes. The final, disjoint diagnostic block remains the decisive validation.

`REUSE=True` reloads a stage already trained in this run. Use a **new run name** when changing samples, architecture, or training settings; this keeps the next notebooks tied to one concrete density model.

In [ ]:
REUSE = True
N_REFERENCE_INTEGRATION = 5_000_000
N_DIAGNOSTIC_REFERENCE = 250_000
NOMINAL_ENSEMBLE = 4
FLOW_MODEL = dict(MODEL_DEFAULTS)
FLOW_TRAINING = dict(TRAIN_DEFAULTS)
RATIO_TRAINING = dict(RATIO_DEFAULTS)

settings = dict(reference_integration=N_REFERENCE_INTEGRATION,
                diagnostic_reference=N_DIAGNOSTIC_REFERENCE,
                nominal_ensemble=NOMINAL_ENSEMBLE,
                flow_model=FLOW_MODEL, flow_training=FLOW_TRAINING,
                ratio_training=RATIO_TRAINING)
(HYBRID / 'training_settings.json').write_text(json.dumps(settings, indent=2))

## 2. Learn a reusable reference density

Train the flow by maximum likelihood on the balanced nominal sample:

$$\mathcal L_{\mathrm{flow}}=-\frac{1}{M}\sum_{j=1}^{M}\log p_{\mathrm{ref},\phi}(x_j).$$

The standardization is fitted on the flow's optimization split, and its Jacobian is included in `log_prob`. The flow's exact normalization comes from its invertible construction. Agreement with the desired mixture must still be checked on independent data.

In [ ]:
flow_training_x = np.concatenate([
    read_sample(RUN, process, partition='flow_train')
    for process in ['signal', 'background']])
reference_flow = train_reference(
    flow_training_x, HYBRID / 'reference.pt', model_config=FLOW_MODEL,
    training_config=FLOW_TRAINING, device=DEVICE, seed=SEED, reuse=REUSE)
del flow_training_x
gc.collect()

saved_flow = torch.load(HYBRID / 'reference.pt', map_location='cpu', weights_only=False)
history = np.asarray(saved_flow['history'])
plt.plot(history[:, 0], label='train')
plt.plot(history[:, 1], label='validation')
plt.xlabel('epoch'); plt.ylabel('mean NLL in standardized coordinates'); plt.legend()
plt.savefig(PLOTS / 'flow_loss.png', bbox_inches='tight')
plt.show()
del saved_flow

In [ ]:
flow_target = np.concatenate([
    read_sample(RUN, process, partition='eval', n=min(125_000, config['samples']['evaluation_per_process']))
    for process in ['signal', 'background']])
flow_diagnostic_x = reference_flow.sample(N_DIAGNOSTIC_REFERENCE, SEED + 1)
fig = plot_shape_closure(flow_target, flow_diagnostic_x,
                         title='Independent balanced-mixture / reference-flow closure')
fig.savefig(PLOTS / 'flow_shape_closure.png', bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(6, 4))
for x, label in [(flow_target, 'held-out balanced mixture'),
                 (flow_diagnostic_x, 'reference flow')]:
    logp = np.sort(reference_flow.log_prob(x))
    ax.plot(logp, np.arange(1, len(logp) + 1) / len(logp), label=label)
ax.set(xlabel='reference log density', ylabel='empirical CDF')
ax.legend(); fig.tight_layout()
fig.savefig(PLOTS / 'flow_log_density_cdf.png')
plt.show()
del flow_target

## 3. Train the nominal process/reference ratios

A classifier with equally normalized class weights minimizes binary cross entropy. Its optimal logit is

$$\ell_c(x)=\log\frac{p_c(x)}{p_{\mathrm{ref}}(x)},\qquad r_c(x)=\exp\ell_c(x).$$

We use the existing `nsbi_common_utils` trainer, with its logit-output option. This is the same classification objective, and avoids converting a numerically saturated sigmoid back into a ratio. `utils.py` returns the arithmetic mean of the four member **ratios**, as in the earlier exercises.

The flow-generated denominator below is used only for ratio training. Neither the normalization bank nor the independent diagnostic reference sample is taken from it. The external reliability plot tests the actual ensemble prediction; the reweighting plots test whether denominator events weighted by the predicted ratio reproduce the target.

In [ ]:
n_ratio = config['samples']['nominal_ratio_per_class']
reference_ratio_training = reference_flow.sample(n_ratio, SEED + 2)
ratio_models = {}
for i, process in enumerate(['signal', 'background']):
    target = read_sample(RUN, process, partition='ratio_train', n=n_ratio)
    ratio_models[process] = train_ratio(
        target, reference_ratio_training, HYBRID / 'ratios' / process,
        members=NOMINAL_ENSEMBLE, seed=SEED + 100 + 10 * i,
        settings=RATIO_TRAINING, reuse=REUSE)
    heldout = read_sample(RUN, process, partition='eval')
    fig = plot_ratio_diagnostics(ratio_models[process], heldout, flow_diagnostic_x,
                                 title=f'{process} / reference: independent validation')
    fig.savefig(PLOTS / f'ratio_{process}.png', bbox_inches='tight')
    plt.show()
    del target, heldout
    gc.collect()
del reference_ratio_training

## 4. Learn the scale-deformation ratios

For each process we train two more classifiers,

$$d_c^\pm(x)=\frac{p_c^{\pm}(x)}{p_c^0(x)}.$$

Their inputs are again the five reconstructed variables; neither $\mu$ nor $\alpha$ enters these fixed-anchor classifiers. The complete raw reference-relative anchors are

$$\widetilde R_c^0=r_c,\qquad \widetilde R_c^- = r_c d_c^-,\qquad \widetilde R_c^+=r_c d_c^+.$$

The deformation ratios are not interpolated in log space. We first construct and normalize these **complete process/reference anchors**, and only then interpolate. We also validate the complete products below: separately good nominal and deformation plots do not by themselves establish good hNDE closure.

In [ ]:
n_variation = config['samples']['variation_ratio_per_class']
for i, process in enumerate(['signal', 'background']):
    # Different latent rows from the varied numerator, whose first rows are used.
    nominal = read_sample(RUN, process, partition='ratio_train')[-n_variation:].copy()
    nominal_eval = read_sample(RUN, process, partition='eval')
    for j, direction in enumerate(['down', 'up']):
        name = f'{process}_{direction}'
        varied = read_sample(RUN, process, partition='ratio_train',
                             anchor=direction, n=n_variation)
        ratio_models[name] = train_ratio(
            varied, nominal, HYBRID / 'ratios' / name,
            members=1, seed=SEED + 200 + 10 * i + j,
            settings=RATIO_TRAINING, reuse=REUSE)
        varied_eval = read_sample(RUN, process, partition='eval', anchor=direction)
        fig = plot_ratio_diagnostics(ratio_models[name], varied_eval, nominal_eval,
                                     title=f'{process}: {direction} / nominal')
        fig.savefig(PLOTS / f'ratio_{name}.png', bbox_inches='tight')
        plt.show()
        del varied, varied_eval
        gc.collect()
    del nominal, nominal_eval

## 5. Normalize the complete anchors once

Draw a fresh integration bank $X_j\sim p_{\mathrm{ref}}$ and define

$$Z_c^k=\frac{1}{M}\sum_j\widetilde R_c^k(X_j),\qquad
R_c^k(x)=\frac{\widetilde R_c^k(x)}{Z_c^k},\quad k\in\{-,0,+\}.$$

The saved array is ordered `[process, anchor] = [signal/background, down/nominal/up]`. Each anchor has sample mean exactly one on this bank. This is **Monte Carlo normalization**, not exact normalization under the continuous flow: the independent closure table next estimates its residual integration error.

For $-1\leq\alpha\leq1$,

$$R_c(x;\alpha)=(1-|\alpha|)R_c^0(x)
 +\max(\alpha,0)R_c^+(x)+\max(-\alpha,0)R_c^-(x).$$

All coefficients are nonnegative and sum to one. Consequently the shapes stay positive and normalized on the integration bank for every allowed $\alpha$. Yields remain $\nu S$ and $B$. We restrict the fit range to $[-1,1]$; extrapolation could produce negative densities and is not part of this model.

In [ ]:
if REUSE and (HYBRID / 'manifest.json').exists():
    hybrid = HybridModel.load(RUN, DEVICE)
else:
    hybrid = HybridModel(RUN, reference_flow, ratio_models)
    manifest = save_hybrid(hybrid, n_reference=N_REFERENCE_INTEGRATION, seed=SEED + 3)
    print('Saved hybrid model:', manifest['model_id'])

integration_anchors = hybrid.reference_anchors()
print('Raw anchor normalizations, rows S/B and columns down/nominal/up:')
print(hybrid.normalization)
print('Integration-bank normalized means:')
print(np.mean(integration_anchors, axis=0))
print('Model ID:', hybrid.model_id)

In [ ]:
# Independent of training AND of the bank used to estimate Z.
check_x = hybrid.sample_reference(N_DIAGNOSTIC_REFERENCE, SEED + 4)
check_anchors = hybrid.anchors(check_x)
rows = []
for c, process in enumerate(['signal', 'background']):
    for k, direction in enumerate(['down', 'nominal', 'up']):
        r = check_anchors[:, c, k]
        mean, se = r.mean(), r.std(ddof=1) / np.sqrt(len(r))
        rows.append(dict(process=process, anchor=direction, mean=mean,
                         standard_error=se, residual=mean - 1,
                         residual_over_check_SE=(mean - 1) / se,
                         effective_sample_size=r.sum()**2 / np.square(r).sum()))
closure = pd.DataFrame(rows)
closure.to_csv(HYBRID / 'independent_normalization_closure.csv', index=False)
display(closure)

for c, process in enumerate(['signal', 'background']):
    for k, direction in enumerate(['down', 'nominal', 'up']):
        target = read_sample(RUN, process, partition='eval', anchor=direction)
        fig = plot_shape_closure(target, check_x, weights=check_anchors[:, c, k],
                                 title=f'Complete {process} {direction} hNDE: held-out closure')
        fig.savefig(PLOTS / f'complete_{process}_{direction}.png', bbox_inches='tight')
        plt.show()
        del target

The reported standard error is the fresh check-bank uncertainty **conditional on the saved normalization constants**. Their own Monte Carlo uncertainty is additional. A large residual or a low effective sample size calls for a larger integration/check bank and another independent check; with $B=10\,000$, small relative normalization errors can matter in a fit.

The projected plots and correlation residuals are useful diagnostics, not a mathematical guarantee of full five-dimensional agreement. The next notebooks include unbinned inference checks using fresh physical simulator events, which can reveal errors these plots miss.

## 6. Reference-sampled, weighted unbinned Asimov fits

The extended likelihood, after omitting parameter-independent reference-density factors, is

$$\log L_H(\nu,\alpha;D,y)=-\nu S-B
 +\sum_{x\in D}\log[\nu S R_S(x;\alpha)+B R_B(x;\alpha)]
 -\frac{(y-\alpha)^2}{2}.$$

For a model-closure Asimov experiment at $(\mu_A,\alpha_A)$, reuse the normalized integration bank with weights

$$w_j=\frac{\mu_A S R_S(X_j;\alpha_A)+B R_B(X_j;\alpha_A)}{M},
\qquad y_A=\alpha_A.$$

These weights sum to the expected count. They replace the event sum in the NLL; there is no binning. We minimize over both $\nu$ and $\alpha$. `fit` checks the two linear-interpolation branches and the kink at zero.

These are **internal closure tests of the good fitted model**, so the fitted parameters should recover the generating values up to optimizer precision. They do not measure model bias on the physical simulator. In later population-response training, the required auxiliary ensemble is $y\sim\mathrm{Uniform}[-2,2]$, whose expected Gaussian-constraint term has the same parameter dependence as $y=0$. Here $y_A=\alpha_A$ is chosen specifically to test model closure at nonzero anchors.

In [ ]:
asimov_rows = []
for mu_A, alpha_A in [(0.5, -0.5), (1.0, 0.0), (1.5, 0.5)]:
    weights = asimov_weights(integration_anchors, mu_A, alpha_A, config)
    result = fit(integration_anchors, auxiliary=alpha_A, config=config, weights=weights)
    asimov_rows.append(dict(mu_A=mu_A, alpha_A=alpha_A,
                            expected_count=mu_A * config['signal_yield'] + config['background_yield'],
                            weight_sum=weights.sum(), fitted_nu=result['nu'],
                            fitted_alpha=result['alpha'], success=result['success']))
asimov_results = pd.DataFrame(asimov_rows)
asimov_results.to_csv(HYBRID / 'asimov_closure.csv', index=False)
display(asimov_results)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
for ax, truth, fitted in [(axes[0], 'mu_A', 'fitted_nu'),
                          (axes[1], 'alpha_A', 'fitted_alpha')]:
    ax.scatter(asimov_results[truth], asimov_results[fitted], label='unbinned Asimov fit')
    limits = [asimov_results[truth].min() - .1, asimov_results[truth].max() + .1]
    ax.plot(limits, limits, 'k--', label='closure')
    ax.set(xlabel=truth, ylabel=fitted)
    ax.legend()
fig.tight_layout()
fig.savefig(PLOTS / 'asimov_parameter_closure.png')
plt.show()

## What is passed to notebook 03?

`hybrid/` contains the learned reference flow, six ratio estimators (the two nominal estimators are four-member ensembles), six normalization constants, and the reference integration bank. `manifest.json` records their model identity. The next notebook changes one explicit mixture parameter $\epsilon$ while keeping this learned model fixed.

Reference-distribution toys are generated by importance resampling from flow proposals. A shared finite bank makes training practical, but it is an approximation to the continuous hNDE distribution. The helper returns the proposal size and effective sample size; fresh proposal banks and fresh physical toys are used to study this approximation separately. The auxiliary observable in all toys is drawn uniformly from $[-2,2]$, as requested, while the likelihood constraint remains Gaussian.